Retrieval Augmented Generation Pipeline for 3GPP Technical Specifications - Gamage S.K.

In [1]:
!pip install -qU langchain langchain-community langchain-groq langchain-chroma langchain-huggingface sentence-transformers pypdf gradio chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [8]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
print("Collections:", client.list_collections())

# Check the default collection
for col in client.list_collections():
    print(f"Collection '{col.name}': {col.count()} documents")

Collections: [Collection(name=langchain)]
Collection 'langchain': 0 documents


In [12]:
import os
import requests
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Download PDF with browser headers to avoid 403
os.makedirs("data", exist_ok=True)
pdf_path = "./data/5G_NR_Spec_138211.pdf"
pdf_url = "https://www.etsi.org/deliver/etsi_ts/138200_138299/138211/18.08.00_60/ts_138211v180800p.pdf"

if not os.path.exists(pdf_path):
    print("Downloading PDF...")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/pdf,*/*",
        "Referer": "https://www.etsi.org/"
    }
    response = requests.get(pdf_url, headers=headers, stream=True, timeout=120)
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))
    downloaded = 0
    with open(pdf_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\r  {downloaded/1e6:.1f} / {total/1e6:.1f} MB", end="")
    print("\nDownload complete.")
else:
    print("PDF already exists, skipping download.")

# 2. Load & split
print("Loading and splitting PDF...")
loader = PyPDFLoader(pdf_path)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} pages.")

# 3. Embed & store
print("Embedding and storing in Chroma DB (may take several minutes)...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)
print(f"Done! {vector_store._collection.count()} chunks stored in ./chroma_db")

  2.5 / 2.5 MB
Download complete.
Loading and splitting PDF...
Created 624 chunks from 171 pages.
Embedding and storing in Chroma DB (may take several minutes)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Done! 624 chunks stored in ./chroma_db


In [13]:
import os
import getpass
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import gradio as gr


# 1. Ensure data directory exists
os.makedirs("data", exist_ok=True)

# NOTE: PDF download step is not implemented here.
# If you need to download the spec, add:
# import urllib.request
# urllib.request.urlretrieve(pdf_url, pdf_path)
pdf_url = "https://www.etsi.org/deliver/etsi_ts/138200_138299/138211/18.08.00_60/ts_138211v180800p.pdf"
pdf_path = "./data/5G_NR_Spec_138211.pdf"

# 2. Secure API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

# 3. Connect to the existing vector DB
print("Connecting to local vector database...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = Chroma(persist_directory="./chroma_db", embedding_function=embedding_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 6})

# 4. Setup LLM
print("Assembling LCEL Pipeline...")

# FIX 1: "openai/gpt-oss-120b" is NOT a valid Groq model.
# Valid Groq models include: "llama-3.3-70b-versatile", "llama3-8b-8192",
# "mixtral-8x7b-32768", "gemma2-9b-it", etc.
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)

system_prompt = (
    "You are a brilliant telecommunications engineering assistant. Answer the user's question "
    "about protocol standards, frequencies, or network architecture using ONLY the context "
    "provided below. If the answer is not contained in the context, admit that you do not know. "
    "Do not hallucinate external facts or specifications.\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Helper to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# FIX 2: Restructured LCEL chain to avoid RunnableParallel dict-arg deprecation
# and fix the context extraction ambiguity.
# The retriever returns docs keyed as "context"; we format them before passing
# to the prompt so {context} in the system prompt gets a clean string.
rag_chain_from_docs = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt_template
    | llm
    | StrOutputParser()
)

# FIX 3: Use keyword arguments for RunnableParallel (not a plain dict)
rag_chain = RunnableParallel(
    context=retriever,
    input=RunnablePassthrough()
).assign(answer=rag_chain_from_docs)

# 5. Define UI Logic
def respond(message: str, history: list) -> str:
    """
    Gradio ChatInterface callback.
    `history` is a list of [user_msg, assistant_msg] pairs.
    """
    if not message.strip():
        return "Please ask a question."

    try:
        response = rag_chain.invoke(message)
        answer = response["answer"]
        source_chunks = response["context"]

        # Build source citations
        unique_sources = set()
        for doc in source_chunks:
            source_file = os.path.basename(doc.metadata.get("source", "Unknown Spec"))
            page = doc.metadata.get("page", 0) + 1
            unique_sources.add(f"- **{source_file}** (Page {page})")

        sources_text = "\n".join(unique_sources) if unique_sources else "- No sources found"
        output = f"{answer}\n\n### 3GPP/ETSI Sources Referenced:\n{sources_text}"
        return output

    except Exception as e:
        return f"An error occurred: {str(e)}"

# 6. Launch Gradio UI
print("Launching Interface...")
demo = gr.ChatInterface(
    fn=respond,
    title="5G & 3GPP Protocol Assistant",
    description=(
        "Ask highly technical questions about 5G NR physical layer specs (3GPP TS 38.211). "
        "Answers are grounded in the ETSI specification document only."
    ),
)

demo.launch(share=True, debug=True)

Connecting to local vector database...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Assembling LCEL Pipeline...
Launching Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9a63a86ab0cc8e7fb6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://9a63a86ab0cc8e7fb6.gradio.live


Evaluation

In [ ]:
!pip install -qU langchain-google-vertexai

In [14]:
import json

def run_comprehensive_evaluation():
    print("Initializing 20-Case 5G NR RAG Evaluation Suite...")

    # 20 Deep-Technical 3GPP TS 38.211 Test Cases
    test_cases = [
        {
            "id": 1,
            "question": "What is the defined subcarrier spacing configuration mu for the physical layer?",
            "keyword": "mu"
        },
        {
            "id": 2,
            "question": "What are the specific parameters or structure for SS/PBCH block type A?",
            "keyword": "FR1"
        },
        {
            "id": 3,
            "question": "How many symbols are contained in a standard 5G NR slot for normal cyclic prefix?",
            "keyword": "14"
        },
        {
            "id": 4,
            "question": "What is the maximum number of subcarriers per resource block (RB) in 5G NR?",
            "keyword": "12"
        },
        {
            "id": 5,
            "question": "What is the purpose and resource mapping structure of the Primary Synchronization Signal (PSS)?",
            "keyword": "synchronization"
        },
        {
            "id": 6,
            "question": "How is the Secondary Synchronization Signal (SSS) positioned relative to the PSS in the frequency domain?",
            "keyword": "subcarriers"
        },
        {
            "id": 7,
            "question": "What are the time-domain resource allocations specified for physical random access channel (PRACH) occasions?",
            "keyword": "preamble"
        },
        {
            "id": 8,
            "question": "What modulation schemes are supported for the Physical Uplink Shared Channel (PUSCH)?",
            "keyword": "QPSK"
        },
        {
            "id": 9,
            "question": "How are Control Resource Sets (CORESET) defined in terms of time and frequency resources?",
            "keyword": "PRB"
        },
        {
            "id": 10,
            "question": "What is the function of Demodulation Reference Signals (DM-RS) for PDSCH?",
            "keyword": "estimation"
        },
        {
            "id": 11,
            "question": "What are the configuration rules for Phase-Tracking Reference Signals (PT-RS) in frequency domain?",
            "keyword": "phase"
        },
        {
            "id": 12,
            "question": "How is Channel State Information Reference Signal (CSI-RS) mapped across antenna ports?",
            "keyword": "ports"
        },
        {
            "id": 13,
            "question": "What is the frame structure duration for a standard 5G radio frame, and how many subframes does it contain?",
            "keyword": "10 ms"
        },
        {
            "id": 14,
            "question": "What subcarrier spacings are supported for higher frequency ranges like FR2-1 and FR2-2?",
            "keyword": "60 kHz"
        },
        {
            "id": 15,
            "question": "How does the extended cyclic prefix configuration affect the number of symbols per slot?",
            "keyword": "12"
        },
        {
            "id": 16,
            "question": "What is a Bandwidth Part (BWP) and how are its parameters constrained by the standard?",
            "keyword": "contiguous"
        },
        {
            "id": 17,
            "question": "What are the multiplexing patterns defined for physical uplink control channel (PUCCH) formats 0 and 1?",
            "keyword": "symbols"
        },
        {
            "id": 18,
            "question": "How is cyclic prefix insertion handled during OFDM baseband signal generation?",
            "keyword": "transform"
        },
        {
            "id": 19,
            "question": "What are the slot formatting rules dictated by TDD configuration parameters?",
            "keyword": "slot"
        },
        {
            "id": 20,
            "question": "How does the specification define the physical resource block (PRB) indexing across different carriers?",
            "keyword": "common"
        }
    ]

    results = []
    passed_retrieval = 0
    passed_faithfulness = 0

    print(f"\nExecuting evaluation across {len(test_cases)} test vectors...\n" + "-"*50)

    for test in test_cases:
        q = test["question"]
        expected = test["keyword"]

        try:
            # Execute RAG chain retrieval and generation
            response = rag_chain.invoke(q)
            answer = response["answer"]
            contexts = [doc.page_content for doc in response["context"]]

            # 1. Retrieval Precision Check (Keyword match in retrieved text chunks)
            kw_found = any(expected.lower() in ctx.lower() for ctx in contexts)
            ret_score = 1.0 if kw_found else 0.0
            if kw_found: passed_retrieval += 1

            # 2. LLM-as-a-Judge Faithfulness Check
            judge_prompt = f"""
            You are an independent AI engineering auditor. Evaluate if the answer is grounded solely in the context.
            Context: {" ".join(contexts)}
            Answer: {answer}

            Does the answer avoid external hallucination and align with the context? Answer strictly with 'YES' or 'NO'.
            """
            judge_res = llm.invoke(judge_prompt).content.strip().upper()
            faith_score = 1.0 if "YES" in judge_res else 0.0
            if faith_score > 0: passed_faithfulness += 1

            results.append({
                "id": test["id"],
                "status": "SUCCESS",
                "retrieval": ret_score,
                "faithfulness": faith_score
            })
            print(f"[{test['id']}/20] Tested: '{q[:40]}...' -> Retrieval: {ret_score} | Faithfulness: {faith_score}")

        except Exception as e:
            results.append({
                "id": test["id"],
                "status": f"ERROR: {str(e)}",
                "retrieval": 0.0,
                "faithfulness": 0.0
            })
            print(f"[{test['id']}/20] FAILED execution.")

    # Calculate aggregate scores for portfolio reporting
    total = len(test_cases)
    ret_acc = (passed_retrieval / total) * 100
    faith_acc = (passed_faithfulness / total) * 100

    print("\n" + "="*50)
    print("      COMPLETED: AGGREGATE EVALUATION METRICS     ")
    print("="*50)
    print(f"Total Test Cases Evaluated : {total}")
    print(f"Retrieval Accuracy (Recall): {ret_acc:.1f}%")
    print(f"Faithfulness Score         : {faith_acc:.1f}%")
    print("="*50)

run_comprehensive_evaluation()

Initializing 20-Case 5G NR RAG Evaluation Suite...

Executing evaluation across 20 test vectors...
--------------------------------------------------
[1/20] Tested: 'What is the defined subcarrier spacing c...' -> Retrieval: 1.0 | Faithfulness: 1.0
[2/20] Tested: 'What are the specific parameters or stru...' -> Retrieval: 0.0 | Faithfulness: 1.0
[3/20] Tested: 'How many symbols are contained in a stan...' -> Retrieval: 1.0 | Faithfulness: 1.0
[4/20] Tested: 'What is the maximum number of subcarrier...' -> Retrieval: 1.0 | Faithfulness: 1.0
[5/20] Tested: 'What is the purpose and resource mapping...' -> Retrieval: 1.0 | Faithfulness: 0.0
[6/20] Tested: 'How is the Secondary Synchronization Sig...' -> Retrieval: 0.0 | Faithfulness: 0.0
[7/20] Tested: 'What are the time-domain resource alloca...' -> Retrieval: 1.0 | Faithfulness: 1.0
[8/20] Tested: 'What modulation schemes are supported fo...' -> Retrieval: 1.0 | Faithfulness: 1.0
[9/20] Tested: 'How are Control Resource Sets (CORESET) ..